# DATASET DOWNLOAD

In [ ]:
from huggingface_hub import login, hf_hub_download
import os

# 1. This will create a text box. Paste your token there and hit Enter.
print("🔑 Logging into Hugging Face...")
#login(token='<>') 

print("Downloading datasets...")
# 2. Now the download will have permission to proceed
hf_hub_download(
    repo_id="Harsh2005/DeepFashion2-Pruned", 
    filename="train_clean.zip", 
    repo_type="dataset", 
    local_dir="/kaggle/working"
)

hf_hub_download(
    repo_id="Harsh2005/DeepFashion2-Pruned", 
    filename="validation_clean.zip", 
    repo_type="dataset", 
    local_dir="/kaggle/working"
)

print("🔓 Unzipping files...")
!unzip -q /kaggle/working/train_clean.zip -d /kaggle/working/train
!unzip -q /kaggle/working/validation_clean.zip -d /kaggle/working/val


import os

# Delete the zip files to free up space
if os.path.exists("/kaggle/working/train_clean.zip"):
    os.remove("/kaggle/working/train_clean.zip")
    print("Deleted train_clean.zip")

if os.path.exists("/kaggle/working/validation_clean.zip"):
    os.remove("/kaggle/working/validation_clean.zip")
    print("Deleted validation_clean.zip")

print("Data is ready!")

🔑 Logging into Hugging Face...


train_clean.zip:   0%|          | 0.00/7.99G [00:00<?, ?B/s]

validation_clean.zip:   0%|          | 0.00/1.32G [00:00<?, ?B/s]

🔓 Unzipping files...
Deleted train_clean.zip
Deleted validation_clean.zip
Data is ready!


In [3]:
!pip install -q segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.8 MB/s eta 0:00:00


In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import json
import numpy as np
from glob import glob
import os
from tqdm import tqdm
import wandb

# Login to W&B
wandb.login()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

CATEGORIES = ["short sleeve top", "trousers", "shorts", "long sleeve top", "skirt"]
CAT_TO_IDX = {cat: i+1 for i, cat in enumerate(CATEGORIES)}
NUM_CLASSES = 6 # 5 classes + 1 background

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dhrramjos (kronpoz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Device: cuda


In [5]:
class SimpleSegDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.json_paths = glob(os.path.join(root_dir, '**/*.json'), recursive=True)

    def __len__(self):
        return len(self.json_paths)

    def __getitem__(self, idx):
        json_path = self.json_paths[idx]
        with open(json_path, 'r') as f:
            data = json.load(f)

        img_path = json_path.replace('.json', '.jpg').replace('annos', 'image')
        if not os.path.exists(img_path):
            img_path = json_path.replace('.json', '.jpg')
        
        image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        mask = np.zeros(image.shape[:2], dtype=np.uint8)
        
        for key, value in data.items():
            if key.startswith('item') and isinstance(value, dict):
                c_name = value.get('category_name')
                if c_name in CAT_TO_IDX and 'segmentation' in value:
                    for poly in value['segmentation']:
                        seg = np.array(poly, dtype=np.int32).reshape(-1, 2)
                        cv2.fillPoly(mask, [seg], CAT_TO_IDX[c_name])
        
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image, mask = transformed['image'], transformed['mask']
            
        return image, mask.long()

train_ts = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_ts = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

train_loader = DataLoader(SimpleSegDataset('/kaggle/working/train/train', train_ts), batch_size=16, shuffle=True, num_workers=4)
val_loader = DataLoader(SimpleSegDataset('/kaggle/working/val/validation', val_ts), batch_size=16, shuffle=False, num_workers=4)

In [6]:
# Initialize W&B run
wandb.init(project="deepfashion-unet", name="unet-run-metrics")

model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=NUM_CLASSES).to(device)

# --- CALCULATE WEIGHTS FROM JSON METADATA ---
print("Calculating class weights directly from JSON metadata...")
class_counts = np.zeros(NUM_CLASSES)

train_json_paths = glob('/kaggle/working/train/train/**/*.json', recursive=True)

# Background is present in every image
class_counts[0] = len(train_json_paths)

# Parse JSONs to get exact instance counts for clothing classes
for json_path in train_json_paths:
    with open(json_path, 'r') as f:
        data = json.load(f)
        for key, value in data.items():
            if key.startswith('item') and isinstance(value, dict):
                c_name = value.get('category_name')
                if c_name in CAT_TO_IDX:
                    class_counts[CAT_TO_IDX[c_name]] += 1

# Calculate inverse frequency weights
weights = 1.0 / (class_counts + 1e-5)
weights = weights / np.max(weights)  
weights_tensor = torch.tensor(weights, dtype=torch.float32).to(device)

print("\nComputed Class Weights (from metadata):")
print(f"Background: {weights_tensor[0]:.4f}")
for i, cat in enumerate(CATEGORIES):
    print(f"{cat}: {weights_tensor[i+1]:.4f}")

criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Calculating class weights directly from JSON metadata...

Computed Class Weights (from metadata):
Background: 0.2139
short sleeve top: 0.4304
trousers: 0.5567
shorts: 0.8421
long sleeve top: 0.8550
skirt: 1.0000


In [ ]:
# --- START TRAINING ---
EPOCHS = 5
LOG_FREQ = 100
global_step = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0
    
    # --- TRAIN ---
    for step, (imgs, masks) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch} Train")):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        current_loss = loss.item()
        train_loss += current_loss
        global_step += 1
        
        if global_step % LOG_FREQ == 0:
            wandb.log({"train/step_loss": current_loss, "global_step": global_step, "epoch": epoch})
        
    # --- VALIDATE ---
    model.eval()
    val_loss = 0
    
    total_tp = torch.zeros(NUM_CLASSES - 1).to(device)
    total_fp = torch.zeros(NUM_CLASSES - 1).to(device)
    total_fn = torch.zeros(NUM_CLASSES - 1).to(device)
    
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f"Epoch {epoch} Val"):
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)
            val_loss += criterion(outputs, masks).item()
            
            preds = outputs.argmax(dim=1)
            
            for c in range(1, NUM_CLASSES):
                idx = c - 1
                pred_c = (preds == c)
                mask_c = (masks == c)
                
                total_tp[idx] += (pred_c & mask_c).sum()
                total_fp[idx] += (pred_c & ~mask_c).sum()
                total_fn[idx] += (~pred_c & mask_c).sum()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    ious = total_tp / (total_tp + total_fp + total_fn + 1e-7)
    dices = (2 * total_tp) / ((2 * total_tp) + total_fp + total_fn + 1e-7)
    
    macro_iou = ious.mean().item()
    macro_dice = dices.mean().item()

    print(f"\nEpoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"-> Macro mIoU: {macro_iou:.4f} | Macro Dice (F1): {macro_dice:.4f}")
    for i, cat in enumerate(CATEGORIES):
        print(f"   {cat}: IoU={ious[i].item():.4f}, Dice={dices[i].item():.4f}")
    print("-" * 50)
    
    log_dict = {
        "train/epoch_loss": avg_train_loss,
        "val/epoch_loss": avg_val_loss,
        "val/macro_mIoU": macro_iou,
        "val/macro_dice_F1": macro_dice,
        "epoch": epoch
    }
    
    for i, cat in enumerate(CATEGORIES):
        log_dict[f"val_IoU/{cat}"] = ious[i].item()
        log_dict[f"val_Dice/{cat}"] = dices[i].item()
        
    wandb.log(log_dict)
    
    model_path = f"/kaggle/working/unet_epoch_{epoch}.pth"
    torch.save(model.state_dict(), model_path)
    wandb.save(model_path) 

wandb.finish()
print("Training complete. Models and metrics uploaded to W&B!")

In [9]:
import wandb
import torch
import os
from tqdm import tqdm

# --- 1. DOWNLOAD WEIGHTS FROM W&B ---
WANDB_RUN_PATH = "kronpoz/deepfashion-unet/qw8zrygu"

print(f"Connecting to W&B run: {WANDB_RUN_PATH}...")
api = wandb.Api()
run = api.run(WANDB_RUN_PATH)

# Find the exact file name in the W&B run
target_file = None
for f in run.files():
    if "unet_epoch_5.pth" in f.name:
        target_file = f.name
        break

if target_file is None:
    raise FileNotFoundError("Could not find unet_epoch_5.pth in the specified W&B run.")

print(f"Downloading {target_file}...")
run.file(target_file).download(root="/kaggle/working", replace=True)

# The file might be downloaded into a subfolder depending on how wandb structured it
DOWNLOADED_PATH = os.path.join("/kaggle/working", target_file)

# Load the downloaded weights
model.load_state_dict(torch.load(DOWNLOADED_PATH, map_location=device))
print(f"Successfully loaded weights from {DOWNLOADED_PATH}")

# --- 2. START NEW RUN FOR EPOCHS 6-10 ---
wandb.init(project="deepfashion-unet", name="unet-run-metrics-epochs-6-to-10")

START_EPOCH = 6
END_EPOCH = 10
LOG_FREQ = 100
global_step = 0

for epoch in range(START_EPOCH, END_EPOCH + 1):
    model.train()
    train_loss = 0
    
    # --- TRAIN ---
    for step, (imgs, masks) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch} Train")):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        current_loss = loss.item()
        train_loss += current_loss
        global_step += 1
        
        if global_step % LOG_FREQ == 0:
            wandb.log({"train/step_loss": current_loss, "global_step": global_step, "epoch": epoch})
        
    # --- VALIDATE ---
    model.eval()
    val_loss = 0
    
    total_tp = torch.zeros(NUM_CLASSES - 1).to(device)
    total_fp = torch.zeros(NUM_CLASSES - 1).to(device)
    total_fn = torch.zeros(NUM_CLASSES - 1).to(device)
    
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f"Epoch {epoch} Val"):
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)
            val_loss += criterion(outputs, masks).item()
            
            preds = outputs.argmax(dim=1)
            
            for c in range(1, NUM_CLASSES):
                idx = c - 1
                pred_c = (preds == c)
                mask_c = (masks == c)
                
                total_tp[idx] += (pred_c & mask_c).sum()
                total_fp[idx] += (pred_c & ~mask_c).sum()
                total_fn[idx] += (~pred_c & mask_c).sum()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    ious = total_tp / (total_tp + total_fp + total_fn + 1e-7)
    dices = (2 * total_tp) / ((2 * total_tp) + total_fp + total_fn + 1e-7)
    
    macro_iou = ious.mean().item()
    macro_dice = dices.mean().item()

    print(f"\nEpoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"-> Macro mIoU: {macro_iou:.4f} | Macro Dice (F1): {macro_dice:.4f}")
    for i, cat in enumerate(CATEGORIES):
        print(f"   {cat}: IoU={ious[i].item():.4f}, Dice={dices[i].item():.4f}")
    print("-" * 50)
    
    log_dict = {
        "train/epoch_loss": avg_train_loss,
        "val/epoch_loss": avg_val_loss,
        "val/macro_mIoU": macro_iou,
        "val/macro_dice_F1": macro_dice,
        "epoch": epoch
    }
    
    for i, cat in enumerate(CATEGORIES):
        log_dict[f"val_IoU/{cat}"] = ious[i].item()
        log_dict[f"val_Dice/{cat}"] = dices[i].item()
        
    wandb.log(log_dict)
    
    model_path = f"/kaggle/working/unet_epoch_{epoch}.pth"
    torch.save(model.state_dict(), model_path)
    wandb.save(model_path) 

wandb.finish()
print("Resumed training complete. Models and metrics uploaded to W&B.")

Connecting to W&B run: kronpoz/deepfashion-unet/qw8zrygu...
Successfully loaded weights from /kaggle/working/working/unet_epoch_5.pth


Epoch 6 Val: 100%|██████████| 1484/1484 [01:39<00:00, 14.97it/s]
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.



Epoch 6 | Train Loss: 0.2283 | Val Loss: 0.3164
-> Macro mIoU: 0.6690 | Macro Dice (F1): 0.8003
   short sleeve top: IoU=0.7293, Dice=0.8435
   trousers: IoU=0.7267, Dice=0.8417
   shorts: IoU=0.6120, Dice=0.7593
   long sleeve top: IoU=0.5983, Dice=0.7487
   skirt: IoU=0.6786, Dice=0.8085
--------------------------------------------------


Epoch 7 Val: 100%|██████████| 1484/1484 [01:40<00:00, 14.76it/s]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.



Epoch 7 | Train Loss: 0.2137 | Val Loss: 0.3069
-> Macro mIoU: 0.6774 | Macro Dice (F1): 0.8063
   short sleeve top: IoU=0.7418, Dice=0.8517
   trousers: IoU=0.7244, Dice=0.8402
   shorts: IoU=0.6033, Dice=0.7526
   long sleeve top: IoU=0.6123, Dice=0.7595
   skirt: IoU=0.7054, Dice=0.8273
--------------------------------------------------


...


Epoch 9 | Train Loss: 0.1869 | Val Loss: 0.3172
-> Macro mIoU: 0.6661 | Macro Dice (F1): 0.7980
   short sleeve top: IoU=0.7346, Dice=0.8470
   trousers: IoU=0.7046, Dice=0.8267
   shorts: IoU=0.5767, Dice=0.7316
   long sleeve top: IoU=0.6131, Dice=0.7602
   skirt: IoU=0.7014, Dice=0.8245
--------------------------------------------------


Epoch 10 Val: 100%|██████████| 1484/1484 [01:39<00:00, 14.93it/s]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.



Epoch 10 | Train Loss: 0.1760 | Val Loss: 0.3349
-> Macro mIoU: 0.6608 | Macro Dice (F1): 0.7941
   short sleeve top: IoU=0.7197, Dice=0.8370
   trousers: IoU=0.7192, Dice=0.8367
   shorts: IoU=0.5771, Dice=0.7319
   long sleeve top: IoU=0.5955, Dice=0.7465
   skirt: IoU=0.6927, Dice=0.8185
--------------------------------------------------


epoch,▁▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆███
global_step,▁▁▁▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇██
train/epoch_loss,█▆▄▂▁
train/step_loss,▂▃▃▃▃▂▂▂▂▄▃▃▃▃▃▁▂▂▂▂▂▂▃▂▃█▂▂▂▂▂▂▃▇▂▁▂▃▂▂
val/epoch_loss,▃▁▅▄█
val/macro_dice_F1,▅█▄▃▁
val/macro_mIoU,▄█▄▃▁
val_Dice/long sleeve top,▂█▆█▁
val_Dice/short sleeve top,▄█▆▆▁
val_Dice/shorts,█▆▄▁▁
+7,...


Resumed training complete. Models and metrics uploaded to W&B.
